##### Copyright 2021 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# End-to-end text preprocessing in TF

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/not_a_real_link"><img src="https://www.tensorflow.org/images/tf_logo_32px.png" />View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/docs/blob/master/tools/templates/notebook.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/docs/blob/master/tools/templates/notebook.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/docs/tools/templates/notebook.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
  <td>
    <a href="https://tfhub.dev/"><img src="https://www.tensorflow.org/images/hub_logo_32px.png" />See TF Hub model</a>
  </td>
</table>

## Overview

Text preprocessing is the end-to-end transformation of raw text into a model’s integer inputs. While in the past models may perform a few preprocessing steps (such as tokenization or string normalizations), we have seen an increasing interest to extract and pretrain models with unsupervised tasks such as BERT’s masked language model (MLM), next sentence prediction (NSP), ALBERT’s sentence order prediction (SOP), etc.

NLP models are often accompanied by several hundreds (if not thousands) of lines of Python text preprocessing code. It becomes increasingly challenging to manage the preprocessing logic without introducing training/serving skew bugs, especially when the model is interacted in multiple stages (e.g. pretraining, fine-tuning, evaluation, inference). For example, using different hyperparameters, tokenization, string preprocessing algorithms or simply packaging model inputs inconsistently could yield hard-to-debug, and disastrous effects to the model. Many of the problems can be alleviated by packaging the preprocessing directly with the model.

Additionally, many existing Python methods write out processed outputs to files on disk and construct TF input pipelines to consume said preprocessed data. This incurs an additional read/write cost and is inconvenient for dynamically changing text preprocessing decisions. Perhaps more importantly, it does not align well with exporting a self-contained model to TF Serving that goes from string input to prediction outputs.

Using TF.Text's preprocessing APIs, users can:

- **Assemble TF input pipelines w/ reusable, well-tested, standard building blocks that transform their text datasets into model inputs.** Being part of the TF graph also enables users to make preprocessing choices dynamically on the fly.
- **Drastically simplify their model’s inputs to just text.** Users will be able to easily expand to new datasets for training, evaluation or inference. Models deployed to TF Serving can start from text inputs and encapsulate the details of preprocessing.
- **Reduce risks of training/serving skew by giving models stronger ownership of the entire preprocessing process.**
- **Reduced complexity and improved input pipeline efficiency** by removing an extra read & write step to transform their datasets and improved efficiency w/ vectorized mapping by processing inputs in batches.

## Setup

In [ ]:
import tensorflow as tf
import tensorflow_text as tf_text
import functools

## Text Preprocessing APIs

### Splitter

`Splitter`s are an abstract class for splitting text. Implementations can include tokenizers, sentence breakers and any class for splitting text. Here are some examples of `Splitter`s and see also [Tokenizing with TF Text](https://www.tensorflow.org/tutorials/tensorflow_text/tokenizers) for examples other tokenizers available in TF.Text. 



#### `BertTokenizer`

Given a vocabulary (generated from the [Wordpiece algorithm](https://www.tensorflow.org/tutorials/tensorflow_text/subwords_tokenizer#optional_the_algorithm)), `BertTokenizer` is a `Splitter` that can tokenize sentences into subwords or wordpieces for the [BERT model](https://github.com/google-research/bert). You can learn more about other subword tokenizers available in TF.Text from [here](https://www.tensorflow.org/tutorials/tensorflow_text/subwords_tokenizer).

Let's look at an example, say that we have the following vocabulary.

In [ ]:
_VOCAB = [
    b"[MASK]",
    b"[RANDOM]",
    b"[CLS]",
    b"[SEP]",
    b"##ack",
    b"##ama",
    b"##gers",
    b"##onge",
    b"##pants",
    b"##uare",
    b"##vel",
    b"##ven",
    b"A",
    b"Bar",
    b"Hates",
    b"Mar",
    b"Ob",
    b"Patrick",
    b"President",
    b"Sp",
    b"Sq",
    b"bob",
    b"box",
    b"has",
    b"highest",
    b"is",
    b"office",
    b"the",
]

_START_TOKEN = _VOCAB.index(b"[CLS]")
_END_TOKEN = _VOCAB.index(b"[SEP]")
_MASK_TOKEN = _VOCAB.index(b"[MASK]")
_RANDOM_TOKEN = _VOCAB.index(b"[RANDOM]")
_VOCAB_SIZE = len(_VOCAB)

lookup_table = tf.lookup.StaticVocabularyTable(
    tf.lookup.KeyValueTensorInitializer(
      _VOCAB,
      tf.range(
          tf.size(_VOCAB, out_type=tf.int64), dtype=tf.int64),
      key_dtype=tf.string,
      value_dtype=tf.int64), 1
)


And we have the following text input:

In [ ]:
text_input =  tf.ragged.constant([
   [b"Sponge bob Squarepants"],
   [b"Barack Obama"],
   [b"Marvel Avengers"]])

We can construct a `BertTokenizer` using the above vocabulary and tokenize the text inputs into a `RaggedTensor` with shape `[batch, num_tokens, num_wordpieces]`.

In [ ]:
bert_tokenizer = tf_text.BertTokenizer(lookup_table, token_out_type=tf.string)
bert_tokenizer.tokenize(text_input)

<tf.RaggedTensor [[[[b'Sp', b'##onge'], [b'bob'], [b'Sq', b'##uare', b'##pants']]], [[[b'Bar', b'##ack'], [b'Ob', b'##ama']]], [[[b'Mar', b'##vel'], [b'A', b'##ven', b'##gers']]]]>

Text output from `BertTokenizer` allows us see how the text is being tokenized, but the model requires integer IDs. We can set the `token_out_type` param to `tf.int64` to obtain integer IDs.

In [ ]:
bert_tokenizer = tf_text.BertTokenizer(lookup_table, token_out_type=tf.int64)
int_output = bert_tokenizer.tokenize(text_input)
int_output

<tf.RaggedTensor [[[[19, 7], [21], [20, 9, 8]]], [[[13, 4], [16, 5]]], [[[15, 10], [12, 11, 6]]]]>

Your preprocessing logic may not require the extra `num_tokens` dimensions, and you can merge the last two dimensions to obtain a output `RaggedTensor` with shape `[batch, num_wordpieces]` like so:

In [ ]:
int_output.merge_dims(-2, -1)

<tf.RaggedTensor [[[19, 7, 21, 20, 9, 8]], [[13, 4, 16, 5]], [[15, 10, 12, 11, 6]]]>

#### `RegexSplitter`

`RegexSplitter` is a subclass of `Splitter` that splits text when a given regex pattern is matched.

In [ ]:
text_input=[
  b"Hi there.\nWhat time is it?\nIt is gametime.",
  b"Who let the dogs out?\nWho?\nWho?\nWho?",
]

sb = tf_text.RegexSplitter(split_regex="\n")
sentences =  sb.split(text_input)
sentences


<tf.RaggedTensor [[b'Hi there.', b'What time is it?', b'It is gametime.'], [b'Who let the dogs out?', b'Who?', b'Who?', b'Who?']]>

#### `StateBasedSentenceBreaker`

  `StateBasedSentenceBreaker` splits text into sentences by using a state machine to
  determine when a sequence of characters indicates a potential sentence break.

  The state machine consists of an "initial state", then transitions to a "collecting
  terminal punctuation state" once an acronym, an emoticon, or terminal punctuation
  (ellipsis, question mark, exclamation point, etc.), is encountered.

  It transitions to the "collecting close punctuation state" when a close punctuation
  (close bracket, end quote, etc.) is found.

  If non-punctuation is encountered in the collecting terminal punctuation or collecting
  close punctuation states, then we exit the state machine, returning false, indicating we have    
  moved past the end of a potential sentence fragment.

In [ ]:
text = [["Hello. Foo bar!"]]
sb = tf_text.StateBasedSentenceBreaker()
split = sb.break_sentences(text)
split

<tf.RaggedTensor [[[b'Hello.', b'Foo bar!']]]>

In [ ]:
split = sb.break_sentences_with_offsets(text)
split

(<tf.RaggedTensor [[[b'Hello.', b'Foo bar!']]]>,
 <tf.RaggedTensor [[[0, 7]]]>,
 <tf.RaggedTensor [[[6, 15]]]>)

### Trimmer

`Trimmer`s trim a list of segments using a predetermined trimming strategy. Removes elements from tensors to ensure that they have a desired maximum size.

When applied to a single tensor, this will mask values from the tensor to  ensure that its size along a specified axis is bounded by a specified maximum  length. 


In [ ]:
segment_a =  tf.ragged.constant([
  [b"hello", b"there"],
  [b"name", b"is"],
  [b"what", b"time", b"is", b"it", b"?"]
])

segment_b = tf.ragged.constant([
  [b"whodis", b"?"],
  [b"bond", b",", b"james", b"bond"],
  [b"5:30", b"AM"]
])



#### `WaterfallTrimmer`

A `WaterfallTrimmer` calculates a drop mask given a budget of the max number of items for each or all batch row. The allocation of the budget is done using a 'waterfall' algorithm. This algorithm allocates quota in a left-to-right manner and fill up the buckets until we run out of budget.

In [ ]:
trimmer = tf_text.WaterfallTrimmer(max_seq_length=[1, 3, 4])
trimmed = trimmer.trim([segment_a, segment_b])
trimmed

[<tf.RaggedTensor [[b'hello'], [b'name', b'is'], [b'what', b'time', b'is', b'it']]>,
 <tf.RaggedTensor [[], [b'bond'], []]>]

When applied to a list of tensors, this will mask values from those tensors to ensure that their *total* length along the specified axis is bounded by a specified maximum length.  E.g.:

In [ ]:
t1 = tf.ragged.constant([[10, 11, 12, 13, 14], [20, 21], [30, 31, 32, 33]])
t2 = tf.ragged.constant([[100, 101], [200, 202, 203], [204, 205]])
trimmer.trim([t1, t2])


[<tf.RaggedTensor [[10], [20, 21], [30, 31, 32, 33]]>,
 <tf.RaggedTensor [[], [200], []]>]

In [ ]:
(t1.row_lengths() + t2.row_lengths()).numpy()  # *total* row length <= 3

array([7, 5, 6])

#### `RoundRobinTrimmer`

Similar to `WaterfallTrimmer`, a `RoundRobinTrimmer` calculates a drop mask given a budget of the max number of items for each or all batch row. However,the allocation of the budget is done using a 'round-robin' algorithm. This algorithm allocates quota to each bucket (in a left-to-right manner) continuously until it runs out of quota.

In [ ]:
trimmer = tf_text.RoundRobinTrimmer(max_seq_length=[2])
trimmed = trimmer.trim([segment_a, segment_b])
trimmed

[<tf.RaggedTensor [[b'hello'], [b'name'], [b'what']]>,
 <tf.RaggedTensor [[b'whodis'], [b'bond'], [b'5:30']]>]

### ItemSelector

`ItemSelector` implementations contain algorithms for selecting items within a `RaggedTensor`. Users of `ItemSelector` implementations can call `get_selection_mask()` to retrieve a bool `RaggedTensor` mask indicating the items that have been selected.


#### FirstNItemSelector

An `ItemSelector` that selects the first `n` items in the batch.

In [ ]:
inputs = tf.ragged.constant([
  [1, 2, 3, 4],
  [100, 200]
])

selector = tf_text.FirstNItemSelector(2)
selected = selector.get_selection_mask(inputs)
selected

<tf.RaggedTensor [[True, True, False, False], [True, True]]>

#### RandomItemSelector

`RandomItemSelector` randomly selects items in a batch subject to restrictions given (max_selections_per_batch, selection_rate and unselectable_ids).

In [ ]:
inputs = tf.ragged.constant([
  [1, 2, 3, 4],
  [100, 200]
])

random_selector = tf_text.RandomItemSelector(
    max_selections_per_batch=2,
    selection_rate=0.2
)
selected = random_selector.get_selection_mask(inputs, axis=1)
selected


<tf.RaggedTensor [[False, True, False, False], [False, True]]>

### MaskValuesChooser


`MaskValuesChooser` encapsulates the logic for deciding the value to assign items that where chosen for masking. The default behavior is consistent with the methodology specified in `Masked LM and Masking Procedure` described in [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/pdf/1810.04805.pdf):

For `mask_token_rate` of the time, replace the item with the `[MASK]` token:

    "my dog is hairy" -> "my dog is [MASK]"
 
For `random_token_rate` of the time, replace the item with a random word:

    "my dog is hairy" -> "my dog is apple"
 
For `1 - mask_token_rate - random_token_rate` of the time, keep the item
unchanged:

    "my dog is hairy" -> "my dog is hairy."




Let's walk through an example. Because `MaskValuesChooser` requires integer inputs, we'll use the above created vocabulary to encode our inputs. Say that we have the following inptus:

In [ ]:
inputs = tf.ragged.constant([
    [b"Sp", b"##onge", b"bob", b"Sq", b"##uare", b"##pants"],
    [b"Bar", b"##ack", b"Ob", b"##ama"],
    [b"Mar", b"##vel", b"A", b"##ven", b"##gers"]]
)
input_ids = lookup_table.lookup(inputs)
input_ids

<tf.RaggedTensor [[19, 7, 21, 20, 9, 8], [13, 4, 16, 5], [15, 10, 12, 11, 6]]>

We can create a `MaskValuesChooser` that masks 80% of the input wordpieces per batch (note that `[MASK] is 0 in our above vocabulary):

In [ ]:
mask_values_chooser = tf_text.MaskValuesChooser(_VOCAB_SIZE, _MASK_TOKEN, 0.8)
mask_values_chooser.get_mask_values(input_ids)

<tf.RaggedTensor [[0, 7, 0, 0, 0, 0], [10, 0, 0, 0], [0, 7, 0, 0, 0]]>

### mask_language_model()


`mask_language_model()` implements the `Masked LM and Masking Procedure` described in [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/pdf/1810.04805.pdf). 

`mask_language_model()` uses an `ItemSelector` to select the items for masking, and a `MaskValuesChooser` to assign the values to the selected items. We can reuse our above input, `random_selector` and `mask_values_chooser` for an example:

In [ ]:
masked_token_ids, masked_pos, masked_ids = tf_text.mask_language_model(
  input_ids,
  item_selector=random_selector, mask_values_chooser=mask_values_chooser)

Let's dive deeper and examine the outputs of `mask_language_model()`. The output of `masked_token_ids` is:

In [ ]:
masked_token_ids

<tf.RaggedTensor [[19, 7, 0, 0, 9, 8], [0, 4, 16, 5], [15, 10, 12, 0, 6]]>

Remember that our input is encoded using a vocabulary. If we decode `masked_token_ids` using our vocabulary, we get:

In [ ]:
tf.gather(_VOCAB, masked_token_ids)

<tf.RaggedTensor [[b'Sp', b'##onge', b'[MASK]', b'[MASK]', b'##uare', b'##pants'], [b'[MASK]', b'##ack', b'Ob', b'##ama'], [b'Mar', b'##vel', b'A', b'[MASK]', b'##gers']]>

Notice that some wordpiece tokens have been replaced with either `[MASK]`, `[RANDOM]` or a different ID value. `masked_pos` output gives us the indices (in the respective batch) of the tokens that have been replaced.

In [ ]:
masked_pos

<tf.RaggedTensor [[2, 3], [0], [3]]>

`masked_ids` gives us the original value of the token.

In [ ]:
 masked_ids

<tf.RaggedTensor [[21, 20], [13], [11]]>

We can again decode the IDs here to get human readable values.

In [ ]:
tf.gather(_VOCAB, masked_ids)

<tf.RaggedTensor [[b'bob', b'Sq'], [b'Bar'], [b'##ven']]>

### combine_segments()

`combine_segments` combines the tokens of one or more input segments to a  single sequence of token values and generates matching segment ids. `combine_segments` may be called after the invocation of a `Truncator`, if the  user seeks to limit segment lengths, and and can be followed up by `pad_model_inputs` to pad the inputs for the model.

In [ ]:
segment_a = tf.ragged.constant([[1, 2],
              [3, 4,],
              [5, 6, 7, 8, 9]])
    
segment_b = tf.ragged.constant([[10, 20,],
              [30, 40, 50, 60,],
              [70, 80]])
expected_combined, expected_ids = tf_text.combine_segments(
  [segment_a, segment_b], start_of_sequence_id=101, end_of_segment_id=102)
expected_combined, expected_ids

(<tf.RaggedTensor [[101, 1, 2, 102, 10, 20, 102], [101, 3, 4, 102, 30, 40, 50, 60, 102], [101, 5, 6, 7, 8, 9, 102, 70, 80, 102]]>,
 <tf.RaggedTensor [[0, 0, 0, 0, 1, 1, 1], [0, 0, 0, 0, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 1, 1, 1]]>)

### pad_model_inputs()


`pad_model_inputs` performs the final packaging of a model's inputs commonly  found in text models. This includes padding out (or simply truncating) to a  fixed-size, 2-dimensional `Tensor` and generating mask `Tensor`s (of the same 2D shape) with values of 0 if the corresponding item is a pad value and 1 if  it is part of the original input.

In [ ]:
inputs={
   "input_ids": tf.ragged.constant([
     [101, 1, 2, 102, 10, 20, 102],
     [101, 3, 4, 102, 30, 40, 50, 60],
     [101, 5, 6, 7, 8, 9, 102, 70],
   ]),
   "segment_ids": tf.ragged.constant([
     [0, 0, 0, 0, 1, 1, 1],
     [0, 0, 0, 0, 1, 1, 1, 1],
     [0, 0, 0, 0, 0, 0, 0, 1],
   ]),
},

results = tf.nest.map_structure(
  functools.partial(tf_text.pad_model_inputs, max_seq_length=10), inputs)
results

({'input_ids': (<tf.Tensor: shape=(3, 10), dtype=int32, numpy=
   array([[101,   1,   2, 102,  10,  20, 102,   0,   0,   0],
          [101,   3,   4, 102,  30,  40,  50,  60,   0,   0],
          [101,   5,   6,   7,   8,   9, 102,  70,   0,   0]], dtype=int32)>,
   <tf.Tensor: shape=(3, 10), dtype=int32, numpy=
   array([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
          [1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
          [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]], dtype=int32)>),
  'segment_ids': (<tf.Tensor: shape=(3, 10), dtype=int32, numpy=
   array([[0, 0, 0, 0, 1, 1, 1, 0, 0, 0],
          [0, 0, 0, 0, 1, 1, 1, 1, 0, 0],
          [0, 0, 0, 0, 0, 0, 0, 1, 0, 0]], dtype=int32)>,
   <tf.Tensor: shape=(3, 10), dtype=int32, numpy=
   array([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
          [1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
          [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]], dtype=int32)>)},)

In [ ]:
padded = {k: v[0] for k, v in results[0].items()}
expected_mask = list(results[0].values())
expected_mask[0]


(<tf.Tensor: shape=(3, 10), dtype=int32, numpy=
 array([[101,   1,   2, 102,  10,  20, 102,   0,   0,   0],
        [101,   3,   4, 102,  30,  40,  50,  60,   0,   0],
        [101,   5,   6,   7,   8,   9, 102,  70,   0,   0]], dtype=int32)>,
 <tf.Tensor: shape=(3, 10), dtype=int32, numpy=
 array([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0]], dtype=int32)>)

## Putting it all together

Using TF.Text's text preprocessing APIs, we can construct a preprocessing
function that can directly transform a user's text dataset into the model's
integer inputs.

Say that we have a dataset of `tf.Example`s containing two text features:


In [ ]:
examples = [
  (b"Sponge bob Squarepants.", b"Barack Obama."),
  (b"Marvel Avengers", b"President is the highest office"),
]

examples = [       
  tf.train.Example(
    features=tf.train.Features(
        feature={
          'text_a': tf.train.Feature(
            bytes_list=tf.train.BytesList(value=[text_a])),          
          'text_b': tf.train.Feature(
            bytes_list=tf.train.BytesList(value=[text_b])),
       })) for text_a,text_b in examples]
examples

[features {
   feature {
     key: "text_a"
     value {
       bytes_list {
         value: "Sponge bob Squarepants."
       }
     }
   }
   feature {
     key: "text_b"
     value {
       bytes_list {
         value: "Barack Obama."
       }
     }
   }
 }, features {
   feature {
     key: "text_a"
     value {
       bytes_list {
         value: "Marvel Avengers"
       }
     }
   }
   feature {
     key: "text_b"
     value {
       bytes_list {
         value: "President is the highest office"
       }
     }
   }
 }]

We can create a preprocessing function (using the above APIs) to transform this text dataset into model inputs for the masked language model task.

In [ ]:
def bert_pretrain_preprocess(vocab_table, example):
  feature_spec = {
    "text_a": tf.io.FixedLenFeature([1], tf.string),
    "text_b": tf.io.FixedLenFeature([1], tf.string),
  }
  features = tf.io.parse_example(example, feature_spec)
  # Input is a string Tensor of documents, shape [batch, 1].
  text_a = features["text_a"]
  text_b = features["text_b"]

  # Tokenize segments to shape [num_sentences, (num_words)] each.
  tokenizer = tf_text.BertTokenizer(
      lookup_table,
      token_out_type=tf.int64)
  segments = [tokenizer.tokenize(text).merge_dims(
      1, -1) for text in (text_a, text_b)]

  # Truncate inputs to a maximum length.
  trimmer = tf_text.RoundRobinTrimmer(max_seq_length=6)
  trimmed_segments = trimmer.trim(segments)

  # Combine segments, get segment ids and add special tokens.
  segments_combined, segment_ids = tf_text.combine_segments(
      trimmed_segments,
      start_of_sequence_id=_START_TOKEN,
      end_of_segment_id=_END_TOKEN)

  # Apply dynamic masking task.
  masked_input_ids, masked_lm_positions, masked_lm_ids = (
      tf_text.mask_language_model(
        segments_combined,
        random_selector,
        mask_values_chooser,
      )
  )

  model_inputs = {
      "input_word_ids": masked_input_ids,
      "input_type_ids": segment_ids,
      "masked_lm_positions": masked_lm_positions,
      "masked_lm_ids": masked_lm_ids,
  }

  # Apply padding
  padded_inputs_and_mask = tf.nest.map_structure(
    functools.partial(tf_text.pad_model_inputs, max_seq_length=6), model_inputs)
  model_inputs = {
      k: padded_inputs_and_mask[k][0] for k in padded_inputs_and_mask
  }
  model_inputs["masked_lm_weights"] = (
      padded_inputs_and_mask["masked_lm_ids"][1])
  model_inputs["input_mask"] = padded_inputs_and_mask["input_word_ids"][1]
  return model_inputs

We can now construct our `tf.data.Dataset` and use our assembled preprocessing function `bert_pretrain_preprocess()` in `Dataset.map()`. This allows us to create an input pipeline for transforming our raw string data into integer inputs and feed directly into our model.

In [ ]:
dataset = tf.data.Dataset.from_tensors([
  example.SerializeToString() for example in examples])
dataset = dataset.map(functools.partial(
    bert_pretrain_preprocess, lookup_table))

next(iter(dataset))

{'input_word_ids': <tf.Tensor: shape=(2, 6), dtype=int64, numpy=
 array([[ 0, 19,  7, 21,  3, 13],
        [ 2, 15, 10,  0,  0, 18]])>,
 'input_type_ids': <tf.Tensor: shape=(2, 6), dtype=int64, numpy=
 array([[0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 1]])>,
 'masked_lm_positions': <tf.Tensor: shape=(2, 6), dtype=int64, numpy=
 array([[0, 7, 0, 0, 0, 0],
        [3, 4, 0, 0, 0, 0]])>,
 'masked_lm_ids': <tf.Tensor: shape=(2, 6), dtype=int64, numpy=
 array([[ 2, 16,  0,  0,  0,  0],
        [12,  3,  0,  0,  0,  0]])>,
 'masked_lm_weights': <tf.Tensor: shape=(2, 6), dtype=int64, numpy=
 array([[1, 1, 0, 0, 0, 0],
        [1, 1, 0, 0, 0, 0]])>,
 'input_mask': <tf.Tensor: shape=(2, 6), dtype=int64, numpy=
 array([[1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1]])>}

## Related Tutorials

[Working with Unicode](https://www.tensorflow.org/tutorials/load_data/unicode)

[Tokenizing with TF Text](https://www.tensorflow.org/tutorials/tensorflow_text/tokenizers)

[Text Loading: Ragged](https://www.tensorflow.org/guide/ragged_tensor)